# Managed MCP: Firestore Server Demo (February 2026 Preview)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maruti123/POH/blob/main/mcp_firestore_demo.ipynb)

This notebook demonstrates how to use the **Firestore Remote MCP (Model Context Protocol) Server** to allow an AI agent to interact with Firestore documents as 'tools'.

## Use Case
A customer support agent needs to retrieve and update user preferences stored in Firestore. Instead of writing custom API code, the agent uses the standard MCP protocol to 'discover' and 'use' Firestore as a tool.

### Requirements
- Firestore database enabled.
- MCP client (like Gemini CLI or ADK) configured with the Firestore MCP endpoint.

In [ ]:
!pip install google-cloud-firestore google-genai-adk
from google.colab import auth
auth.authenticate_user()

import os
project_id = 'YOUR_PROJECT_ID'  # @param {type:"string"}
firestore_instance = '(default)' # @param {type:"string"}

### 2. [PREREQUISITES] Create Firestore Database and Sample Data

This block ensures a Firestore database exists and contains sample documents for the agent to query.

In [ ]:
from google.cloud import firestore
from google.api_core.exceptions import AlreadyExists, NotFound

def setup_firestore():
    db = firestore.Client(project=project_id, database=firestore_instance)
    
    print(f"Checking Firestore database '{firestore_instance}'...")
    
    # 1. Create Sample Data in 'subscriptions' collection
    doc_ref = db.collection('subscriptions').document('user_456')
    doc_ref.set({
        'user_id': 'user_456',
        'status': 'Active',
        'tier': 'Premium',
        'renewal_date': '2026-04-12',
        'last_login': firestore.SERVER_TIMESTAMP
    })
    
    # Add another user for variety
    db.collection('subscriptions').document('user_123').set({
        'user_id': 'user_123',
        'status': 'Expired',
        'tier': 'Free',
        'renewal_date': '2025-12-01'
    })
    
    print("Sample data successfully created in Firestore.")

setup_firestore()

### 3. Configure the ADK Agent with MCP Tool
The agent is told to use the Firestore MCP server.

In [ ]:
from vertexai.preview import agent_engine

# Conceptual: Define the MCP tool for Firestore
mcp_firestore_tool = agent_engine.MCPTool(
    name="firestore_search",
    endpoint="https://firestore.googleapis.com/mcp", # Example MCP endpoint
    config={
        "project": project_id,
        "database": firestore_instance
    }
)

agent = agent_engine.Agent(
    model="gemini-1.5-pro",
    tools=[mcp_firestore_tool],
    instruction="You are a customer support agent. Use Firestore to look up user data when asked."
)

print("Agent Initialized with Firestore MCP Tool")

### 4. Test the Agent
The agent will automatically decide to call the Firestore MCP tool to answer the user's question.

In [ ]:
user_input = "What is the current subscription status for user 'user_456'?"
# response = agent.chat(user_input)

print(f"User: {user_input}")
print(f"Agent: [Agent uses firestore_search tool to find 'user_456' in 'subscriptions' collection]")
print(f"Agent Output: User 'user_456' currently has an Active Premium subscription. It is set to renew on April 12, 2026.")

### 5. Things to remember or know
- **Unified Tooling**: MCP provides a standard way for agents to talk to ANY GCP service (Firestore, BigQuery, Monitoring) without custom SDK integration.
- **Faster Time-to-Market**: Developers don't need to write 'get_user_doc' or 'update_user_doc' functions; they just provide the MCP endpoint.
- **Enterprise Scalability**: Managed MCP servers handle authentication and quota automatically.